# ClaimWise Insurance Prediction — 13. Decision Tree Classifier (Classification Extension)

## 1. Objective

Add a **Decision Tree Classifier** next to the Logistic Regression classifier from notebook 12 and
compare both on the same derived target. The existing **Decision Tree Regressor**
(notebook 05, `Src/decision_tree_regressor_model.py`) is untouched — this is a separate model for a
separate, documented target.

| Item | Location |
|---|---|
| Features | `Dataset/06_split_X_train.csv`, `06_split_X_test.csv` (existing preprocessing) |
| Classification target | `loss_high`, derived exactly as in notebook 12 |
| Shared helpers | `Src/academic_common.py` |

Metrics: confusion matrix, precision, recall, F1 — computed on held-out test predictions.

## 2. Import Libraries

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Walk upwards from the notebook folder until the ClaimWise project is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "Dataset").is_dir() and (candidate / "Src").is_dir():
            return candidate
    raise FileNotFoundError(f"ClaimWise project root not found above: {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / "Dataset"
SRC_DIR = PROJECT_ROOT / "Src"
MODELS_DIR = PROJECT_ROOT / "Models"
OUTPUTS_DIR = PROJECT_ROOT / "Outputs"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root  :", PROJECT_ROOT)
print("Dataset folder:", DATASET_DIR)
print("Src folder    :", SRC_DIR)

from academic_common import (
    CLASSIFICATION_DIR,
    CLASS_TARGET,
    build_classification_target,
    classification_metrics,
    ensure_dirs,
    plot_confusion_matrix,
    save_figure,
    save_metric_rows,
    style_axes,
)
from model_training_utils import load_prepared_data

ensure_dirs()

Project root  : /Users/padaltiruvinayak/Desktop/Credit ML
Dataset folder: /Users/padaltiruvinayak/Desktop/Credit ML/Dataset
Src folder    : /Users/padaltiruvinayak/Desktop/Credit ML/Src


In [3]:
# Src/model_training_utils.py switches matplotlib to the non-interactive Agg backend so the
# command-line scripts can run without a display. Switch back to the inline backend here so
# that every figure below is rendered inside this notebook.
%matplotlib inline

## 3. Classification Target (same derivation as notebook 12)

Threshold = median of the 40,000 training losses, positive class = high-loss claim. Recomputing it
through the same shared helper guarantees the two classification notebooks use identical labels.

In [4]:
y_train_cls, y_test_cls, target_info = build_classification_target()
print("Derived target :", CLASS_TARGET)
print("Threshold      :", round(target_info["threshold"], 4))
print("Train counts   :", target_info["train_counts"])
print("Test counts    :", target_info["test_counts"])

Derived target : loss_high
Threshold      : 2118.86
Train counts   : {'typical (0)': 20000, 'high (1)': 20000}
Test counts    : {'typical (0)': 5037, 'high (1)': 4963}


In [5]:
X_train, X_test, y_train, y_test = load_prepared_data()
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("label balance train:", target_info["train_counts"], "| test:", target_info["test_counts"])
assert len(X_train) == len(y_train_cls) and len(X_test) == len(y_test_cls)
assert list(X_train.columns) == list(X_test.columns)
print("Data alignment checks passed.")

X_train: (40000, 130) | X_test: (10000, 130)
label balance train: {'typical (0)': 20000, 'high (1)': 20000} | test: {'typical (0)': 5037, 'high (1)': 4963}
Data alignment checks passed.


## 4. Train the Decision Tree Classifier

Same hyper-parameter shape as the project's tree regressor (`max_depth=20`, `min_samples_leaf=2`,
`random_state=42`) so the two tree variants are comparable — but this is a **classifier** fitted on
`loss_high`, a different estimator from `DecisionTreeRegressor`.

In [6]:
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier(max_depth=20, min_samples_leaf=2, random_state=42)
clf.fit(X_train, y_train_cls)
print(clf)
print("tree depth:", clf.get_depth(), "| leaves:", clf.get_n_leaves())

DecisionTreeClassifier(max_depth=20, min_samples_leaf=2, random_state=42)
tree depth: 20 | leaves: 4078


In [7]:
y_pred = clf.predict(X_test)
metrics = classification_metrics(y_test_cls, y_pred)
print("Decision Tree Classifier — test-set classification metrics")
for name, value in metrics.items():
    print(f"  {name:10s} {value:.6f}")

save_metric_rows(
    CLASSIFICATION_DIR / "decision_tree_classifier_metrics.csv",
    "Decision Tree Classifier",
    metrics,
)

Decision Tree Classifier — test-set classification metrics
  accuracy   0.688200
  precision  0.706237
  recall     0.636510
  f1         0.669563


,model,accuracy,precision,recall,f1
0,Decision Tree Classifier,0.6882,0.706237,0.63651,0.669563


In [8]:
from sklearn.metrics import classification_report
print(classification_report(
    y_test_cls, y_pred,
    target_names=["typical loss (0)", "high loss (1)"],
    digits=4,
))

                  precision    recall  f1-score   support

typical loss (0)     0.6736    0.7391    0.7048      5037
   high loss (1)     0.7062    0.6365    0.6696      4963

        accuracy                         0.6882     10000
       macro avg     0.6899    0.6878    0.6872     10000
    weighted avg     0.6898    0.6882    0.6873     10000



In [9]:
cm_path = plot_confusion_matrix(
    y_test_cls,
    y_pred,
    "Decision Tree Classifier — confusion matrix (test set)",
    CLASSIFICATION_DIR / "decision_tree_confusion_matrix.png",
)
from IPython.display import Image, display as display_image
display_image(filename=str(cm_path))

from sklearn.metrics import confusion_matrix
tn, fp, fn, tp = confusion_matrix(y_test_cls, y_pred, labels=[0, 1]).ravel()
print(f"TN={tn} FP={fp} FN={fn} TP={tp}")

TN=3723 FP=1314 FN=1804 TP=3159


In [10]:
importances = pd.DataFrame(
    {"feature": X_train.columns, "importance": clf.feature_importances_}
).sort_values("importance", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.bar(importances["feature"], importances["importance"], color="#D4512A")
style_axes(ax, "Decision Tree Classifier — 10 most important features", "", "importance")
imp_path = save_figure(fig, CLASSIFICATION_DIR / "decision_tree_classifier_top_features.png")
display(importances.reset_index(drop=True))
print("Saved:", imp_path)

,feature,importance
0,cat80,0.199172
1,cont14,0.059478
2,cat12,0.050013
3,cont7,0.038234
4,cat101,0.031178
5,cat100,0.029145
6,cont2,0.028239
7,cat103,0.027151
8,cont8,0.026856
9,cat112,0.025866


Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/classification/decision_tree_classifier_top_features.png


## 5. Logistic Regression vs Decision Tree Classifier

Both classifiers are read back from the metrics CSVs written during this run, so the comparison
table and chart cannot drift from the executed numbers. The image below is the classification
metrics comparison (`accuracy`, `precision`, `recall`, `F1`) — regression metrics are deliberately
not shown here.

In [11]:
logreg = pd.read_csv(CLASSIFICATION_DIR / "logistic_regression_metrics.csv")
dectree = pd.read_csv(CLASSIFICATION_DIR / "decision_tree_classifier_metrics.csv")
summary = pd.concat([logreg, dectree], ignore_index=True)[
    ["model", "accuracy", "precision", "recall", "f1"]
]
summary_path = CLASSIFICATION_DIR / "classification_metrics_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved:", summary_path)
display(summary.round(4))

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/classification/classification_metrics_summary.csv


,model,accuracy,precision,recall,f1
0,Logistic Regression,0.7583,0.7828,0.7101,0.7446
1,Decision Tree Classifier,0.6882,0.7062,0.6365,0.6696


In [12]:
plot = summary.set_index("model")[["accuracy", "precision", "recall", "f1"]]
ax = plot.plot(kind="bar", figsize=(9, 5), color=["#D4512A", "#E58A3A", "#3F7D4A", "#18181B"], rot=0)
style_axes(ax.get_figure().axes[0], "Classification metrics comparison — test set (higher is better)",
           "", "score")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=8, padding=2)
ax.set_ylim(0, 1.08)
ax.legend(loc="lower right")
fig = ax.get_figure()
cmp_path = save_figure(fig, CLASSIFICATION_DIR / "classification_metric_comparison.png")
print("Saved:", cmp_path)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/classification/classification_metric_comparison.png


## 6. Conclusion

Actual results of this executed run (10,000 held-out test rows, positive class = high-loss claim):

| Model | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| Logistic Regression | 0.7583 | 0.7828 | 0.7101 | 0.7446 |
| Decision Tree Classifier | 0.6882 | 0.7062 | 0.6365 | 0.6696 |

Artifacts: `Outputs/classification/decision_tree_classifier_metrics.csv`,
`decision_tree_confusion_matrix.png`, `decision_tree_classifier_top_features.png`,
`classification_metrics_summary.csv`, `classification_metric_comparison.png`.

Regression note: the Decision Tree **Regressor** of notebook 05 and its MAE/MSE/RMSE/R² results are
unchanged; classification metrics are kept in this separate section on purpose.